In [9]:
import sys
sys.path.append('../src')

import os
import json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
import pandas as pd

from dataset import HAM10000Dataset
from models import Lens
from tools import train, test, score

device = torch.device("cuda:1")
print("Device:", device)

Device: cuda:1


In [10]:
df = pd.read_csv('../data/HAM10000_metadata_cleaned.csv')

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['dx']
)
train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv',   index=False)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset_7 = HAM10000Dataset('../data/train.csv', '../data/HAM10000_images', transform=train_transform)
test_dataset_7  = HAM10000Dataset('../data/test.csv',  '../data/HAM10000_images', transform=test_transform)
train_dataset_2 = HAM10000Dataset('../data/train.csv', '../data/HAM10000_images', transform=train_transform, binary=True)
test_dataset_2  = HAM10000Dataset('../data/test.csv',  '../data/HAM10000_images', transform=test_transform,  binary=True)

def get_sampler(binary=False):
    if binary:
        counts  = train_df['dx'].map(lambda x: 'mel' if x == 'mel' else 'other').value_counts()
        weights = train_df['dx'].map(lambda x: 1.0 / counts['mel'] if x == 'mel' else 1.0 / counts['other']).values.copy()
    else:
        counts  = train_df['dx'].value_counts()
        weights = train_df['dx'].map(lambda x: 1.0 / counts[x]).values.copy()
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

def initialize_loaders(batch_size=64, num_workers=8, binary=False):
    td      = train_dataset_2 if binary else train_dataset_7
    vd      = test_dataset_2  if binary else test_dataset_7
    sampler = get_sampler(binary)
    train_loader = DataLoader(td, batch_size=batch_size, sampler=sampler,  num_workers=num_workers)
    test_loader  = DataLoader(vd, batch_size=batch_size, shuffle=False,    num_workers=num_workers)
    return train_loader, test_loader

mel_idx = train_dataset_7.label_encoder.classes_.tolist().index('mel')
print(f"mel_idx: {mel_idx}")

Train: 5805, Test: 1452
mel_idx: 4


In [11]:
def run(epochs=20, lr=0.0001, dropout=0.5, binary=False, mel_weight=1.0, use_meta=False, **model_kwargs):

    train_loader, test_loader = initialize_loaders(binary=binary)

    n_classes = 2 if binary else 7
    model     = Lens(n_classes=n_classes, dropout=dropout, **model_kwargs).to(device)
    mid = 1 if binary else mel_idx
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": [],
               "TP": [], "FP": [], "FN": [], "TN": [],
               "auc": [], "conf_true": [], "conf_false": []}

    for epoch in range(epochs):
        train_loss, train_acc                      = train(model, optimizer, criterion, train_loader, device, use_meta)
        test_loss,  test_acc                       = test( model, criterion,            test_loader,  device, use_meta)
        TP, FP, FN, TN, auc, conf_true, conf_false = score(model, test_loader, device, mid, use_meta)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)
        history["TP"].append(TP)
        history["FP"].append(FP)
        history["FN"].append(FN)
        history["TN"].append(TN)
        history["auc"].append(auc)
        history["conf_true"].append(conf_true)
        history["conf_false"].append(conf_false)

        if epoch == epochs - 1:
            sens = TP / (TP + FN) if (TP + FN) > 0 else 0
            spec = TN / (TN + FP) if (TN + FP) > 0 else 0
            print(f"  → test {test_acc*100:.1f}% | sens {sens*100:.1f}% spec {spec*100:.1f}% | auc {auc:.3f}", flush=True)

    return model, history

In [12]:
os.makedirs('../results', exist_ok=True)
results = {}

for resnet_size in [18, 34]:
    for binary in [False, True]:
        for n_unfreeze in [0, 1, 2]:
            for n_linear in [1, 2, 3]:
                name      = f"R{resnet_size}_{'bin' if binary else '7kl'}_u{n_unfreeze}_l{n_linear}"
                json_path = f'../results/{name}_history_A.json'
                total     = 2 * 2 * 3 * 3
                done      = len(results)

                if os.path.exists(json_path):
                    print(f"[skip] {name}", flush=True)
                    with open(json_path) as f:
                        results[name] = json.load(f)
                    continue

                print(f"[{done+1}/{total} — {(done+1)/total*100:.0f}%] {name}", flush=True)

                model, history = run(
                    epochs      = 20,
                    binary      = binary,
                    resnet_size = resnet_size,
                    n_unfreeze  = n_unfreeze,
                    n_linear    = n_linear,
                )
                results[name] = history

                with open(json_path, 'w') as f:
                    json.dump(history, f)
                torch.save(model.state_dict(), f'../results/{name}_weights_A.pth')
                print(f"  → saved {name}", flush=True)

print("DONE!", flush=True)

[1/36 — 3%] R18_7kl_u0_l1


  → test 67.0% | sens 52.1% spec 88.5% | auc 0.855
  → saved R18_7kl_u0_l1
[2/36 — 6%] R18_7kl_u0_l2
  → test 70.0% | sens 51.2% spec 90.0% | auc 0.874
  → saved R18_7kl_u0_l2
[3/36 — 8%] R18_7kl_u0_l3
  → test 70.0% | sens 55.4% spec 88.0% | auc 0.861
  → saved R18_7kl_u0_l3
[4/36 — 11%] R18_7kl_u1_l1
  → test 82.5% | sens 52.1% spec 93.4% | auc 0.904
  → saved R18_7kl_u1_l1
[5/36 — 14%] R18_7kl_u1_l2
  → test 82.7% | sens 55.4% spec 94.4% | auc 0.926
  → saved R18_7kl_u1_l2
[6/36 — 17%] R18_7kl_u1_l3
  → test 83.0% | sens 44.6% spec 95.6% | auc 0.901
  → saved R18_7kl_u1_l3
[7/36 — 19%] R18_7kl_u2_l1
  → test 83.1% | sens 49.6% spec 94.5% | auc 0.902
  → saved R18_7kl_u2_l1
[8/36 — 22%] R18_7kl_u2_l2
  → test 82.8% | sens 56.2% spec 94.5% | auc 0.912
  → saved R18_7kl_u2_l2
[9/36 — 25%] R18_7kl_u2_l3
  → test 82.9% | sens 65.3% spec 91.7% | auc 0.906
  → saved R18_7kl_u2_l3
[10/36 — 28%] R18_bin_u0_l1
  → test 74.7% | sens 82.6% spec 74.0% | auc 0.873
  → saved R18_bin_u0_l1
[11/36 —